In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

sns.set(style="white", color_codes=True)

from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Loading Dataset
iris = datasets.load_iris()

df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['target'] = iris.target

print("First 5 rows of dataset:")
print(df.head())

# Checking dataset balance
df.groupby('target').size().plot(kind='barh')
plt.show()

# Distance Function
def dis(a, b, p=1):
    l = len(a)
    d = 0

    for i in range(l):
        d += abs(a[i] - b[i]) ** p

    d = d ** (1/p)

    return d

# Testing one sample point
X = df.drop('target', axis=1)
y = df.target

test_pt = [4.8, 2.7, 2.5, 0.7]

distances = []

for i in X.index:
    a = dis(test_pt, X.iloc[i])
    distances.append(a)

dists = pd.DataFrame(data=distances, index=X.index, columns=['dist'])

print("\nDistance DataFrame:")
print(dists.head())

# Sorting nearest neighbors
def knn_sort(k, dists):
    return dists.sort_values(by='dist')[:k]

sorted_dists = knn_sort(5, dists)

print("\nTop 5 nearest distances:")
print(sorted_dists)

count_set = {}

for i in sorted_dists.index:
    if y[i] not in count_set:
        count_set[y[i]] = 1
    else:
        count_set[y[i]] += 1

print("\nPredicted Class:")
print(max(count_set))

# Splitting dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=1
)

# Feature Scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# KNN Function
def KNN(X_train, X_test, y_train, y_test, k, p):

    y_predict = []

    for test_pt in X_test:

        distances = []

        for i in X_train:
            a = dis(test_pt, i, p)
            distances.append(a)

        dists = pd.DataFrame(
            data=distances,
            index=y_train.index,
            columns=['dist']
        )

        sorted_dists = knn_sort(k, dists)

        count_set = {}

        for i in sorted_dists.index:

            if y_train[i] not in count_set:
                count_set[y_train[i]] = 1
            else:
                count_set[y_train[i]] += 1

        y_predict.append(max(count_set))

    y_actual = y_test.tolist()

    accr = 0

    print("\nCorrect and Wrong Predictions:\n")

    for i in range(len(y_actual)):

        print("Actual:", y_actual[i],
              " Predicted:", y_predict[i])

        if y_actual[i] == y_predict[i]:
            accr += 1

    accuracy = accr / len(y_actual)

    return accuracy

# Calling the function
accuracy = KNN(
    X_train,
    X_test,
    y_train,
    y_test,
    5,
    1
)

print("\nAccuracy:", accuracy)

# Checking accuracy for different K values
accuracies = []

for i in range(1, 100):
    accuracies.append(
        KNN(X_train, X_test, y_train, y_test, i, 1)
    )

print("\nBest Accuracy:")
print(max(accuracies))

# Plotting graph
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(range(1, 100), accuracies)

ax.set_xlabel('# of Nearest Neighbors (k)')
ax.set_ylabel('Accuracy (%)')

plt.show()

# Data Visualization

# Scatter Plot
df.plot(
    kind="scatter",
    x="sepal length (cm)",
    y="sepal width (cm)"
)

plt.show()

# Joint Plot
sns.jointplot(
    x="sepal length (cm)",
    y="sepal width (cm)",
    data=df,
    height=5
)

plt.show()

# Box Plot
sns.boxplot(
    x="target",
    y="petal length (cm)",
    data=df
)

plt.show()

# Box Plot + Strip Plot
cx = sns.boxplot(
    x="target",
    y="petal length (cm)",
    data=df
)

cx = sns.stripplot(
    x="target",
    y="petal length (cm)",
    data=df,
    jitter=True,
    edgecolor="gray"
)

plt.show()

# Violin Plot
sns.violinplot(
    x="target",
    y="petal length (cm)",
    data=df
)

plt.show()